# Naive Bayes จำแนกดาวรีวิวร้านอาหารไทย (Wongnai)

ทายจำนวนดาว (1-5) จากข้อความรีวิว ด้วย **Naive Bayes** ตามสไลด์ *08 Bayesian Learning* (สไลด์ 29)

ขั้นตอน: อ่านข้อมูล → แบ่งเรียน/ทดสอบ → ตัดคำไทย → Vocabulary → **Learn** → **Classify** → Performance test → Heat map → Word cloud

รายละเอียดโจทย์อยู่ใน `Ice.md`

> รันจากโฟลเดอร์โปรเจกต์นี้ (ต้องมี `fonts/Sarabun-Regular.ttf`) ส่วน `data/` ถ้าไม่มีจะโหลดจาก GitHub ให้เองในข้อ 1
> ไลบรารี: numpy, matplotlib, pythainlp, wordcloud

In [ ]:
import csv
import os
import random
import time
import urllib.request
import zipfile

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from pythainlp.tokenize import word_tokenize
from pythainlp.corpus.common import thai_stopwords
from wordcloud import WordCloud

FONT = 'fonts/Sarabun-Regular.ttf'          # ฟอนต์ไทยสำหรับวาดภาพ (ไม่เกี่ยวกับการคำนวณ)
font_manager.fontManager.addfont(FONT)
plt.rcParams['font.family'] = font_manager.FontProperties(fname=FONT).get_name()

random.seed(0)                              # ผลซ้ำเดิมทุกครั้ง

## 1. อ่านข้อมูล
ใช้ `w_review_train.csv` จาก [wongnai-corpus](https://github.com/wongnai/wongnai-corpus) (40,000 รีวิว มีดาว 1-5)
ไฟล์คั่นด้วย `;` ไม่มีบรรทัดหัวตาราง และบางรีวิวขึ้นบรรทัดใหม่ จึงอ่านด้วย `csv` module

In [ ]:
CSV = 'data/w_review_train.csv'
URL = 'https://raw.githubusercontent.com/wongnai/wongnai-corpus/master/review/review_dataset.zip'

if not os.path.exists(CSV):                 # ยังไม่มีไฟล์ -> โหลดจาก GitHub
  os.makedirs('data', exist_ok=True)
  urllib.request.urlretrieve(URL, 'data/review_dataset.zip')
  with zipfile.ZipFile('data/review_dataset.zip') as z:
    z.extract('w_review_train.csv', 'data')
  os.remove('data/review_dataset.zip')

csv.field_size_limit(10**9)
reviews = []                                # ข้อความรีวิว
stars = []                                  # ดาว 1-5
with open(CSV, encoding='utf-8', newline='') as f:
  for text, star in csv.reader(f, delimiter=';'):
    reviews.append(text)
    stars.append(int(star))
stars = np.array(stars)

print('จำนวนรีวิว =', len(reviews))
for s in range(1, 6):
  print(s, 'ดาว :', np.sum(stars == s), 'รีวิว')